# Download files from dbGaP or SRA with DRS

This notebook demonstrates how to use the DRS API, together with the helper API called IDX, to programmatically download authorized files from a dbGaP study or SRA accession.

## To use this notebook

1. **Get an NIH RAS passport.** You should have either an NIH PIV login or an eRA Commons account. If you have a place to download your NIH RAS passport, download one first. You can also go to the [dbGaP Power User Portal](https://www.ncbi.nlm.nih.gov/gap/power-user-portal/) to download a dbGaP RAS passport.
2. **Try the defaults.** Choose dbGaP for the example study `phs000710`, or SRA for the example accession `SRR21790641`.
3. **Use your own authorized studies.** If you have previously used the **File Selector** for dbGaP phenotype files, the **Run Selector** for dbGaP sequence files in SRA, or the **Image Selector** for studies with image files, this notebook demonstrates how DRS can be used to programmatically download all authorized files with a small Python script.

> **Timing note:** The notebook should take about **1 minute** to run through the setup. The final file download for our test dbGaP Study takes about **1 minute**.

## 1. Configuration and setup

The next cell imports the local setup helper, prompts for dbGaP or SRA data, an accession, your RAS passport path, and a safe output directory. Press Enter to accept each displayed default.

The defaults are `gap/phs000710` and `sra/SRR21790641`. Setup and manifest retrieval should take about 1 minute; the final download may take a few additional minutes.

In [ ]:
# ruff: noqa: T201
# step 1: configuration and setup
import json
import logging
import time
from datetime import datetime
from pathlib import Path

import requests
from helpers import configure_download

config = configure_download()

## 2. Retrieve the manifest
### Example response
You will get output similar to the example below. Each entry under `contents` represents a file and is keyed by its DRS object ID. The exact DRS IDs, filenames, and sizes will vary by study.
```json
{
  "contents": {
    "NCBI.DRS.0.6cc8591f6718564f4bffe58fa48ab3ac.0.23f0c72777dcfa92da91a8977b66a12e": {
      "md5": "23f0c72777dcfa92da91a8977b66a12e",
      "name": "aa_release/NLM/CloudTesting/phs000710v2/p1/phenotype/phs000710.v2.pht003986.v1.p1.Cloud_Testing_Pedigree.MULTI.txt.gz",
      "size": 446
    },
    "NCBI.DRS.0.0fa17f4130e82fd77331e1b5d2ae9b08.0.d4f7e12a282e4b4856a814ae219c2610": {
      "md5": "d4f7e12a282e4b4856a814ae219c2610",
      "name": "phs000710.v2.pht003986.v1.Cloud_Testing_Pedigree.data_dict.xml",
      "size": 910
    }
  }
}
```
### Retrieve DRS object metadata
Each key in `contents` is a DRS object ID. You can append a DRS object ID to the DRS `/objects/` endpoint to retrieve its metadata.

**Click the URL below to view the DRS metadata for the first file:**

[https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/NCBI.DRS.0.6cc8591f6718564f4bffe58fa48ab3ac.0.23f0c72777dcfa92da91a8977b66a12e](https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/NCBI.DRS.0.6cc8591f6718564f4bffe58fa48ab3ac.0.23f0c72777dcfa92da91a8977b66a12e)

The general URL pattern is:
`https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/<DRS_OBJECT_ID>`

The returned DRS metadata includes the object's available access methods. The download code later in this notebook uses an access method to obtain the file's access URL and download the file.

In [ ]:
# step 2.
# Below is to download the manifest of dataset files given a selected study or accession (ex. dbGaP study or SRA Project Accession).

resp = requests.get(f"{config.idx_base_url}/compound-object-manifest/{config.source}", timeout=60)
resp.raise_for_status()  # Exception if there are errors
j = resp.json()

contents = j.get("response", {}).get("contents", {})
print("Manifest retrieval complete.")
print(f"Study/source: {config.source}")
print(f"The manifest contains {len(contents):,} DRS objects/files.")
j

## 3. Define a function to download one DRS object
This utility function downloads one DRS object to a local file. It tries each access method returned by the DRS API until one succeeds. For each successful file, it reports the DRS access endpoint, start/end timestamps, and elapsed time.
As shown in Step 2, you can retrieve the DRS object metadata using a URL such as the one below. **Click the URL to see the DRS metadata returned for this object:**

[https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/NCBI.DRS.0.0fa17f4130e82fd77331e1b5d2ae9b08.0.d4f7e12a282e4b4856a814ae219c2610](https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/NCBI.DRS.0.0fa17f4130e82fd77331e1b5d2ae9b08.0.d4f7e12a282e4b4856a814ae219c2610)

The returned DRS metadata includes an `access_id` for each available access method. Using the `access_id`, you can construct the access endpoint used to obtain the file's download URL. For controlled-access files, you must POST the user's RAS passport with this request.

The access endpoint has this general form:
```text
https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/<DRS_OBJECT_ID>/access/<ACCESS_ID>
```
For the DRS object above, an example is:
```text
https://locate-beta.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/NCBI.DRS.0.0fa17f4130e82fd77331e1b5d2ae9b08.0.d4f7e12a282e4b4856a814ae219c2610/access/8aa1886f8f5cc0a0e62b75dc6b6c2a583c22fb8d71d3e4a465ec509c0fd14f74
```
The function below performs these steps programmatically for each DRS object.

In [ ]:
# step 3.
#
def seconds_since(start_time: datetime) -> float | int:
    """Return the number of seconds since start_time"""
    return (datetime.now().astimezone() - start_time).total_seconds()


def dl_drs_id_to_file(id_: str, filename: Path, file_number: int, total_files: int) -> float | None:
    """Download one DRS object and return elapsed seconds."""
    object_start = datetime.now().astimezone()
    print()
    print("-" * 80)
    print(f"INFO: Downloading file {file_number:,} of {total_files:,}: {filename.name}")
    print(f"INFO: Started at: {object_start:%Y-%m-%d %H:%M:%S %Z}")
    print("-" * 80)

    resp = requests.get(f"{config.drs_base_url}/objects/{id_}", timeout=60)
    try:
        resp.raise_for_status()
    except requests.exceptions.HTTPError:
        logging.exception(
            f"File {file_number:,}: failed to retrieve DRS object metadata for {filename} from {id_}"
            f"elapsed time before failure: {seconds_since(object_start):.2f} seconds"
        )
        return None

    methods = resp.json()["access_methods"]
    for method in methods:
        access_id = method["access_id"]
        access_endpoint = f"{config.drs_base_url}/objects/{id_}/access/{access_id}"
        try:
            resp = requests.post(
                access_endpoint,
                data=json.dumps({"passports": [config.passport_text]}),
                headers={"Content-type": "application/json"},
                timeout=60,
            )
            resp.raise_for_status()
            url = resp.json()["url"]
            with requests.get(url, stream=True) as response:
                response.raise_for_status()
                with open(filename, "wb") as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            file.write(chunk)

            object_end = datetime.now().astimezone()
            elapased_sec = seconds_since(object_start)
            logging.info(f"File {file_number:,}: download complete: {filename}")
            logging.info(f"File {file_number:,}: DRS object ID: {id_}")
            logging.info(f"File {file_number:,}: DRS access endpoint: {access_endpoint}")
            logging.info(f"File {file_number:,}: finished at: {object_end:%Y-%m-%d %H:%M:%S %Z}")
            logging.info(f"File {file_number:,}: download time: {elapased_sec:.2f} seconds")
            return elapased_sec
        except Exception:
            logging.exception(
                f"File {file_number:,}: failed to download {filename} from {id_}; trying the next access method"
            )
            continue

    logging.error(
        f"File {file_number:,}: could not download DRS ID {id_} to {filename}"
        f"elapsed time before failure: {seconds_since(object_start):.2f} seconds"
    )
    return None


print("Download function is ready.")

## 4. Download all files for which the DRS IDs are listed in the manifest file.

The manifest contains one DRS object for each file. The loop below downloads every object to `config.output_directory`.

In [ ]:
RETRY_WAIT_SECONDS = 15
MAX_DOWNLOAD_RETRIES = 3

download_start = datetime.now().astimezone()
download_times: list[int | float] = []
download_failures = 0

total_objects = len(j["response"]["contents"])
print()
print("=" * 80)
print(f"INFO: Starting download of {total_objects:,} files")
print(f"INFO: Overall download started at: {download_start:%Y-%m-%d %H:%M:%S %Z}")
print(f"INFO: Output directory: {config.output_directory.resolve()}")
print(f"INFO: Failed downloads will be retried up to {MAX_DOWNLOAD_RETRIES} times")
print(f"INFO: Retry wait time: {RETRY_WAIT_SECONDS} seconds")
print("=" * 80)

for file_number, (id_, v) in enumerate(j["response"]["contents"].items(), start=1):
    name = Path(v["name"]).name

    elapsed_seconds = None

    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 2):
        elapsed_seconds = dl_drs_id_to_file(
            id_,
            config.output_directory / name,
            file_number,
            total_objects,
        )

        if elapsed_seconds is not None:
            break

        # +1 because the first attempt is not considered a retry.
        retries_used = attempt

        if retries_used <= MAX_DOWNLOAD_RETRIES:
            print(
                f"WARNING: File {file_number} of {total_objects} failed. "
                f"Waiting {RETRY_WAIT_SECONDS} seconds before retry "
                f"{retries_used} of {MAX_DOWNLOAD_RETRIES}..."
            )
            time.sleep(RETRY_WAIT_SECONDS)

    if elapsed_seconds is None:
        download_failures += 1
        print(
            f"ERROR: File {file_number} of {total_objects} failed after "
            f"{MAX_DOWNLOAD_RETRIES + 1} total attempts: {name}"
        )
    else:
        download_times.append(elapsed_seconds)

download_end = datetime.now().astimezone()
total_elapsed_seconds = (download_end - download_start).total_seconds()

print()
print("=" * 80)
print("INFO: All downloads processed.")
print(f"INFO: Overall download ended at: {download_end:%Y-%m-%d %H:%M:%S %Z}")
print(f"INFO: Processed: {total_objects:,} DRS objects/files")
print(f"INFO: Successfully downloaded: {len(download_times):,}")
print(f"INFO: Failed downloads: {download_failures:,}")
print(
    f"INFO: Total download time: {total_elapsed_seconds:.2f} seconds "
    f"({total_elapsed_seconds / 60:.2f} minutes)"
)
print(f"INFO: Downloaded files are in: {config.output_directory.resolve()}")
print("=" * 80)